# 04b — Consistency Analysis

Test LLM prediction stability by running the **same model** on the **same 100 loans** multiple times.

**Goal:** Measure how many predictions flip across runs due to LLM non-determinism.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    evaluate_predictions, RESULTS_DIR
)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
API_PROVIDER = "gemini"
MODEL_NAME   = "gemini-2.5-flash"  # Use the model from 04a that performed best
LABEL        = "Gemini 2.5 Flash"
INCLUDE_DESC = True                # Test with desc (best condition from 04a)
N_RUNS       = 3

## Load Data

In [ ]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)

print(f"Sample size: {len(llm_sample)}")
print(f"Running {N_RUNS} runs of {LABEL} ({'with' if INCLUDE_DESC else 'without'} desc)")

## Run Multiple Times

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def run_one(run_idx):
    return run_llm_experiment(
        llm_sample,
        api_provider=API_PROVIDER,
        model_name=MODEL_NAME,
        include_desc=INCLUDE_DESC,
        label=f"{LABEL} Run {run_idx + 1}",
    )

# Run all N_RUNS in parallel
with ThreadPoolExecutor(max_workers=N_RUNS) as pool:
    runs = list(pool.map(run_one, range(N_RUNS)))

print(f"\nAll {N_RUNS} runs complete.")

## Stability Analysis

In [ ]:
# Build prediction matrix: rows = samples, columns = runs
pred_matrix = pd.DataFrame({
    f'run_{i+1}': r['predictions'] for i, r in enumerate(runs)
})
pred_matrix['actual'] = y_true

# Per-sample agreement
run_cols = [c for c in pred_matrix.columns if c.startswith('run_')]
pred_matrix['all_agree'] = pred_matrix[run_cols].nunique(axis=1) == 1
pred_matrix['majority_vote'] = pred_matrix[run_cols].mode(axis=1)[0].astype(int)
pred_matrix['majority_correct'] = (pred_matrix['majority_vote'] == pred_matrix['actual']).astype(int)

n_stable = pred_matrix['all_agree'].sum()
n_unstable = len(pred_matrix) - n_stable

print(f"Prediction Stability:")
print(f"  Stable (all {N_RUNS} runs agree): {n_stable}/100 ({n_stable}%)")
print(f"  Unstable (at least one flip):     {n_unstable}/100 ({n_unstable}%)")
print(f"\nMajority vote accuracy: {pred_matrix['majority_correct'].mean()*100:.1f}%")

In [ ]:
# Per-run metrics comparison
run_metrics = []
for i, r in enumerate(runs):
    m = r['metrics'].copy()
    m['run'] = i + 1
    run_metrics.append(m)

metrics_df = pd.DataFrame(run_metrics).set_index('run')
print("Per-run metrics:")
print(metrics_df.to_string())

print(f"\nAccuracy range: {metrics_df['accuracy'].min()*100:.1f}% - {metrics_df['accuracy'].max()*100:.1f}%")
print(f"Accuracy std:   {metrics_df['accuracy'].std()*100:.2f}%")
print(f"CO F1 range:    {metrics_df['f1_charged_off'].min():.3f} - {metrics_df['f1_charged_off'].max():.3f}")

In [ ]:
# Inspect unstable predictions
unstable = pred_matrix[~pred_matrix['all_agree']].copy()
if len(unstable) > 0:
    print(f"\nUnstable predictions ({len(unstable)} samples):")
    print("="*60)
    for idx, row in unstable.iterrows():
        actual = 'Fully Paid' if row['actual'] == 1 else 'Charged Off'
        preds = [row[c] for c in run_cols]
        pred_str = ', '.join(['FP' if p == 1 else 'CO' for p in preds])
        majority = 'FP' if row['majority_vote'] == 1 else 'CO'
        correct = 'correct' if row['majority_correct'] else 'wrong'
        print(f"  Sample {idx}: Actual={actual} | Runs=[{pred_str}] | Majority={majority} ({correct})")
else:
    print("All predictions were perfectly stable across runs!")

## Export Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save prediction matrix
pred_matrix.to_csv(f"{RESULTS_DIR}/04b_consistency_predictions.csv", index=False)

# Save per-run metrics
metrics_df.to_csv(f"{RESULTS_DIR}/04b_consistency_metrics.csv")

print(f"Results saved to {RESULTS_DIR}/")
print(f"\nSummary: {n_stable}% stable, accuracy range {metrics_df['accuracy'].min()*100:.1f}%-{metrics_df['accuracy'].max()*100:.1f}%")